# Heart Disease Classification

## End-to-End Machine Learning Project

This project uses the uploaded **heart-disease dataset** to build and
evaluate classification models for predicting the target outcome.

### Project Workflow

**Data → Exploratory Analysis → Train/Test Split → Model Training →
Model Comparison → Hyperparameter Tuning → Cross-Validation → Evaluation**

### Models Used

- K-Nearest Neighbors (KNN)
- Logistic Regression
- Random Forest Classifier

The project follows the same overall structure and modelling approach as
the original notebook provided for this project.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    cross_val_score
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Import the Dataset

The dataset used here is the exact CSV uploaded for this project.

In [ ]:
df = pd.read_csv(r"/mnt/data/heart-disease (1).csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (303, 14)


   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  ca  thal  target
0   63    1   3       145   233    1        0      150      0      2.3      0   0     1       1
1   37    1   2       130   250    0        1      187      0      3.5      0   0     2       1
2   41    0   1       130   204    0        0      172      0      1.4      2   0     2       1
3   56    1   1       120   236    0        1      178      0      0.8      2   0     2       1
4   57    0   0       120   354    0        1      163      1      0.6      2   0     2       1

## 2. Understand the Data

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum())

print("\nData types:")
print(df.dtypes)

Column names:
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

Missing values:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0

Data types:
age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca            int64
thal          int64
target        int64


## 3. Features and Target

The target column is separated from the input features. Numerical columns
are used for the machine learning models.

In [ ]:
target_column = df.columns[-1] if "target" not in df.columns else "target"

X = df.drop(target_column, axis=1)
X = X.select_dtypes(include=np.number)

y = df[target_column]

print("Target column:", target_column)
print("Number of features:", X.shape[1])
print("Number of samples:", X.shape[0])

Target column: target
Number of features: 13
Number of samples: 303


## 4. Train/Test Split

The data is split into training and testing sets. The test set is kept
separate for final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 242
Testing samples: 61


## 5. Train Different Classification Models

I compare three commonly used classification algorithms before selecting
a stronger model for tuning.

In [ ]:
models = {
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

model_results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    model_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    })

results = pd.DataFrame(model_results).sort_values(
    "Accuracy",
    ascending=False
)

results.round(3)

              Model  Accuracy  Precision  Recall  F1 Score
      Random Forest     0.836      0.858   0.836     0.831
Logistic Regression     0.803      0.813   0.803     0.800
                KNN     0.590      0.591   0.590     0.591

## 6. Random Forest Hyperparameter Tuning

The original project also explored improving the Random Forest model.
Here I use `RandomizedSearchCV` with 5-fold cross-validation.

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_grid,
    n_iter=10,
    cv=5,
    random_state=42,
    scoring="accuracy",
    n_jobs=-1
)

search.fit(X_train, y_train)

print("Best parameters:")
print(search.best_params_)
print("\nBest cross-validation score:", round(search.best_score_, 3))

Best parameters:
{'n_estimators': 50, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_depth': 5}

Best cross-validation score: 0.839


## 7. Evaluate the Tuned Model

In [ ]:
tuned_model = search.best_estimator_
y_pred = tuned_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print("Tuned Random Forest Performance")
print("--------------------------------")
print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1 Score : {f1:.3f}")

Tuned Random Forest Performance
--------------------------------
Accuracy : 0.836
Precision: 0.858
Recall   : 0.836
F1 Score : 0.831


## 8. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[19  9]
 [ 1 32]]


## 9. Classification Report

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

              precision    recall  f1-score   support

           0       0.95      0.68      0.79        28
           1       0.78      0.97      0.86        33

    accuracy                           0.84        61
   macro avg       0.87      0.82      0.83        61
weighted avg       0.86      0.84      0.83        61


## 10. Cross-Validation

Cross-validation gives a more reliable estimate of how consistently the
model performs across different data splits.

In [ ]:
cv_scores = cross_val_score(
    tuned_model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Cross-validation scores:")
print(np.round(cv_scores, 3))

print("Mean CV accuracy:", round(cv_scores.mean(), 3))
print("Standard deviation:", round(cv_scores.std(), 3))

Cross-validation scores:
[0.82  0.902 0.852 0.85  0.767]

Mean CV accuracy: 0.838
Standard deviation: 0.044


## 11. Feature Importance

Random Forest provides feature importance values that can help identify
which variables contributed most to the predictions.

In [ ]:
feature_importance = pd.Series(
    tuned_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

display(feature_importance.to_frame("Importance").head(10))

          Importance
cp          0.184728
thalach     0.127176
oldpeak     0.124427
thal        0.120239
ca          0.101665
exang       0.087438
slope       0.079831
age         0.055007
chol        0.047265
trestbps    0.031847

## 12. ROC-AUC

ROC-AUC is included because this dataset has a binary target.

In [ ]:
probabilities = tuned_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, probabilities)

print("ROC-AUC:", round(auc, 3))

ROC-AUC: 0.904


## Conclusion

This project demonstrates an end-to-end classification workflow on the
uploaded heart disease dataset.

### Key Steps Completed

- Loaded and explored the dataset
- Separated features and target
- Created training and testing sets
- Compared KNN, Logistic Regression and Random Forest
- Tuned Random Forest hyperparameters
- Evaluated the tuned model
- Used confusion matrix and classification report
- Performed 5-fold cross-validation
- Examined feature importance

### Final Takeaway

The project shows how a classification model can be trained, compared,
tuned and evaluated rather than relying on a single accuracy score.

The next improvement would be to experiment with feature engineering,
additional models and more systematic hyperparameter tuning.